## How to use the code

This script is built to be fully interactive so you can test it on any dataset without having to hardcode variables every single time. Just hit the **Run All** button on the cells below and follow the terminal prompts:

1. **Choose your input mode:**
   * Type `y` if you want to manually drop in your own test sequences.
   * Type `n` to just run the default sequence block from a youtube video (Check Reference).
2. **Enter sequences (if you chose `y`):**
   * Type each DNA sequence one by one and hit `ENTER`.
   * When you are finished adding sequences, literally just type `DONE` to lock them all in.
3. **Set the Motif Length ($l$):**
   * The code will ask if you have a specific "Target Motif" you're hunting for (e.g., `GATTACA`). If you type it in, the code automatically sets $l$ to the length of that word and does a quick flex by scoring how closely its final answer matches your target.
   * If you don't have a target and just want to blindly search for a pattern of a certain size, hit `ENTER` to skip, and then type the integer length (e.g., `6`).

**What to expect in the output:**
Because `show_steps=True` is ON, the code is gonna dump a huge **Step-by-Step Breakdown** during its very first iteration. It includes the raw count matrix, the pseudocount probability derivation, and the literal mathematical substitution for the $R(w)$ weights so that we can crosscheck. After this, it spits out the final Log-Odds Matrix and the best aligned motifs across all sequences. (For CrossChecking purposes)

### Engine

In [ ]:
import random
import math
from typing import List, Dict, Tuple

def get_background(sequences: List[str]) -> Dict[str, float]:
    """Calculates the background nucleotide frequencies across all sequences."""
    counts = {'A': 1.0, 'C': 1.0, 'G': 1.0, 'T': 1.0}
    total_nucleotides = 4.0

    for seq in sequences:
        for nt in seq.upper():
            if nt in counts:
                counts[nt] += 1
                total_nucleotides += 1

    return {k: v / total_nucleotides for k, v in counts.items()}


    # WHY ?:
    # I started the counts at 1.0 instead of 0 (Laplace smoothing).
    # Tbh, if a specific nucleotide literally never appears in the background,
    # leaving it at 0 will zero out the whole probability chain later. (from the yt videp)
    # Slapping a pseudocount on everything ensures the math doesn't blow up with divide-by-zero errors.




def create_profile(sequences: List[str], positions: List[int], exclude_idx: int, motif_length: int) -> List[Dict[str, float]]:
    """Builds the position weight matrix (Profile M') from the sequences."""
    profile = [{'A': 1.0, 'C': 1.0, 'G': 1.0, 'T': 1.0} for _ in range(motif_length)]

    for i, seq in enumerate(sequences):
        if i == exclude_idx:
            continue

        start_pos = positions[i]
        motif = seq.upper()[start_pos:start_pos + motif_length]

        for j in range(motif_length):
            if motif[j] in profile[j]:
                profile[j][motif[j]] += 1

    for j in range(motif_length):
        column_total = sum(profile[j].values())
        for nt in profile[j]:
            profile[j][nt] /= column_total

    return profile


    # WHY ?:
    # Notice the `exclude_idx` check.
    # I'm explicitly leaving out the current sequence we are evaluating.
    # If we include it, it's basically data leakage and the model gets biased.
    # Leaving it out keeps the profile honest.
    # Also added pseudocounts here again to prevent zero-probabilities.



def score_l_mer(segment: str, profile: List[Dict[str, float]], background: Dict[str, float], motif_length: int) -> float:
    """Calculates the odds ratio Q(w) / P(w) for a single sliding window."""
    p_motif = 1.0
    p_bg = 1.0

    for j in range(motif_length):
        nt = segment[j]
        if nt in background:
            p_motif *= profile[j][nt]
            p_bg *= background[nt]
        else:
            return 0.0

    return p_motif / p_bg if p_bg > 0 else 0.0


    # WHY ?:
    # I score using the odds ratio (Motif Prob / Background Prob)
    # instead of just taking the raw motif probability.
    # Why? Because some DNA sequences are naturally GC-rich or AT-rich.
    # Dividing by the background normalizes the score
    # so the algorithm doesn't get tricked by naturally repeating junk DNA.


def sample_new_position(sequence: str, profile: List[Dict[str, float]], background: Dict[str, float], motif_length: int, show_math: bool = False) -> int:
    """Slides a window across the sequence, scores each l-mer, and picks a new start position."""
    sequence = sequence.upper()
    weights = []

    if show_math:
        print("\n" + "="*60)
        print("STEP-BY-STEP MATH SUBSTITUTION (Scoring Windows)")
        print("Formula: Weight R(w) = Q(w) / P(w)")
        print(f"Target Sequence: {sequence}")
        print("-" * 60)

    for i in range(len(sequence) - motif_length + 1):
        segment = sequence[i:i + motif_length]
        weight = score_l_mer(segment, profile, background, motif_length)

        # Dumping the math for the manual trace (cross-checking)
        if show_math:
            q_sub, p_sub = [], []
            q_val, p_val = 1.0, 1.0
            for j in range(motif_length):
                nt = segment[j]
                q_prob = profile[j][nt] if nt in profile[j] else 0
                p_prob = background[nt] if nt in background else 0
                q_sub.append(f"{q_prob:.4f}")
                p_sub.append(f"{p_prob:.4f}")
                q_val *= q_prob
                p_val *= p_prob

            print(f"• Window [{i}]: '{segment}'")
            print(f"  Q('{segment}') = {' * '.join(q_sub)} = {q_val:.6e}")
            print(f"  P('{segment}') = {' * '.join(p_sub)} = {p_val:.6e}")
            print(f"  Weight (Q/P) = {q_val:.6e} / {p_val:.6e} = {weight:.4f}\n")

        weights.append(weight)

    total_weight = sum(weights)
    if total_weight == 0:
        return random.randint(0, len(sequence) - motif_length)

    random_threshold = random.random()
    cumulative_prob = 0.0

    for i, w in enumerate(weights):
        cumulative_prob += (w / total_weight)
        if random_threshold <= cumulative_prob:
            return i

    return len(sequence) - motif_length


    # WHY ?:
    # I explicitly didn't use a greedy approach (picking the max weight every time)
    # because Gibbs sampling will absolutely get trapped in a local optimum. (from deeplearning)
    # Using a roulette wheel selection based on normalized weights
    # gives the algorithm a chance to explore and eventually lock onto the true global motif.



def compute_total_score(sequences: List[str], positions: List[int], motif_length: int, background: Dict[str, float]) -> float:
    """Calculates the overall log-odds convergence score L(a1.....an)."""
    profile = create_profile(sequences, positions, -1, motif_length)
    total_score = 0.0

    for i, seq in enumerate(sequences):
        pos = positions[i]
        segment = seq.upper()[pos:pos + motif_length]
        ratio = score_l_mer(segment, profile, background, motif_length)

        if ratio > 0:
            total_score += math.log2(ratio)

    return total_score


    # WHY ?:
    # I convert the final alignment scores into log-odds (base 2). Adding logs
    # is computationally way more stable than multiplying a bunch of tiny
    # floating-point probabilities, which usually just causes an underflow error.



def print_log_odds_matrix(profile: List[Dict[str, float]], background: Dict[str, float], motif_length: int):
    """Utility to print the final position weight matrix."""
    print("\nLog-Odds Matrix R(w):")
    print("Nt |" + "".join([f"{i+1:^8}" for i in range(motif_length)]))
    print("-" * (5 + 8 * motif_length))

    for nt in ['A', 'C', 'G', 'T']:
        row_str = f"{nt}  |"
        for j in range(motif_length):
            ratio = profile[j][nt] / background[nt]
            log_odds = math.log2(ratio) if ratio > 0 else -99.99
            row_str += f"{log_odds:>7.2f} "
        print(row_str)


def print_step_by_step_matrix(sequences: List[str], positions: List[int], exclude_idx: int, motif_length: int):
    """Utility to show the raw count matrix and probability derivation for a manual trace."""
    bg_counts = {'A': 0, 'C': 0, 'G': 0, 'T': 0}
    motif_counts = [{'A': 0, 'C': 0, 'G': 0, 'T': 0} for _ in range(motif_length)]

    for i, seq in enumerate(sequences):
        if i == exclude_idx:
            continue
        pos = positions[i]
        motif = seq.upper()[pos:pos + motif_length]

        for j in range(motif_length):
            if motif[j] in motif_counts[j]:
                motif_counts[j][motif[j]] += 1

        bg_chars = seq.upper()[:pos] + seq.upper()[pos + motif_length:]
        for nt in bg_chars:
            if nt in bg_counts:
                bg_counts[nt] += 1


    print("STEP-BY-STEP BREAKDOWN (First Iteration)")
    print("Goal: Count residue occurrences for each position for all unchosen sequences.")


    for i, seq in enumerate(sequences):
        if i == exclude_idx:
            print(f"• {seq}  <-- EXCLUDED (Target for sampling)")
        else:
            pos = positions[i]
            motif = seq[pos:pos+motif_length]
            print(f"• {seq[:pos]}[{motif}]{seq[pos+motif_length:]}")

    print("\nRaw Count Matrix (Matches lecture slides before pseudocounts):")
    header = "| Nt | bg0 |" + "".join([f" {j+1:^3} |" for j in range(motif_length)])
    print(header)
    print("-" * len(header))

    for nt in ['A', 'C', 'G', 'T']:
        row_str = f"|  {nt} | {bg_counts[nt]:^3} |"
        for j in range(motif_length):
            row_str += f" {motif_counts[j][nt]:^3} |"
        print(row_str)

    print("\nProbability Derivation (Adding +1 Pseudocount):")
    for j in range(motif_length):
        col_total = sum(motif_counts[j].values())
        print(f"Pos {j+1}: " + ", ".join([f"P({nt})=({motif_counts[j][nt]}+1)/({col_total}+4)={((motif_counts[j][nt] + 1) / (col_total + 4)):.4f}" for nt in ['A', 'C', 'G', 'T']]))


def run_gibbs_sampler(sequences: List[str], motif_length: int, num_restarts: int = 25, patience: int = 50, show_steps: bool = False) -> Tuple[List[int], List[str], float, List[Dict[str, float]]]:
    """Core Gibbs Sampling loop with random restarts."""
    n = len(sequences)
    background = get_background(sequences)

    global_best_score = float('-inf')
    global_best_positions = []
    global_best_profile = []

    first_step_shown = False

    for restart in range(num_restarts):

        positions = [random.randint(0, len(seq) - motif_length) for seq in sequences]

        local_best_score = float('-inf')
        local_best_positions = list(positions)
        patience_counter = 0

        while True:
            for h in range(n):

                trigger_math = False
                if show_steps and not first_step_shown:
                    print_step_by_step_matrix(sequences, positions, h, motif_length)
                    trigger_math = True
                    first_step_shown = True

                profile = create_profile(sequences, positions, h, motif_length)
                positions[h] = sample_new_position(sequences[h], profile, background, motif_length, show_math=trigger_math)

            current_score = compute_total_score(sequences, positions, motif_length, background)

            if current_score > local_best_score:
                local_best_score = current_score
                local_best_positions = list(positions)
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                break

        if local_best_score > global_best_score:
            global_best_score = local_best_score
            global_best_positions = list(local_best_positions)
            global_best_profile = create_profile(sequences, global_best_positions, -1, motif_length)

    final_motifs = [sequences[i].upper()[global_best_positions[i]:global_best_positions[i] + motif_length] for i in range(n)]
    return global_best_positions, final_motifs, global_best_score, global_best_profile, background


    # WHY ?:
    # Since the starting positions are chosen completely at random, the algorithm is highly sensitive to initialization.
    # It can easily converge on a garbage motif just because it started in a weird spot.
    # Looping it a bunch of times (num_restarts) guarantees we eventually stumble onto the true biological alignment.




### Input/Output

In [ ]:
if __name__ == "__main__":

    print("Gibbs Sampler - Assignment Build")

    choice = input("Do you want to type your own sequences manually? If not the code will use a test sequence.. (y/n): ").strip().lower()

    if choice == 'y':
        exam_sequences = []
        print("\nEnter your sequences one by one [Use only A, T, G, C].")
        print("Type 'DONE' when you are finished.")
        while True:
            seq = input("Enter Sequence: ").strip().upper()
            if seq == 'DONE':
                break
            if seq:
                exam_sequences.append(seq)

        target_input = input("\nEnter the target motif if you know it (or hit ENTER to skip): ").strip().upper()
        if target_input:
            target_motif = target_input
            l_mer_length = len(target_motif)
        else:
            target_motif = None
            l_mer_length = int(input("Enter the motif length (l) you want to search for: "))

    else:
        print("\nLoading the default test sequences...") # You can add sequence here too
        exam_sequences = [
            "ACCATGACAG",
            "GAGTATACCT",
            "CATGCTTACT",
            "CGGAATGCAT"
        ]
        target_motif = "GATTACA" # Make sure you enter the target if youre changing the default sequence from the code
        l_mer_length = len(target_motif)

    print(f"\nInitiating Gibbs Sampler for {l_mer_length}-mers...")

    best_positions, final_motifs, best_score, best_profile, bg_model = run_gibbs_sampler(
        sequences=exam_sequences,
        motif_length=l_mer_length,
        num_restarts=30,
        show_steps=True
    )

    print(f"\nFinal Convergence Score L: {best_score:.4f}")

    print_log_odds_matrix(best_profile, bg_model, l_mer_length)

    print("\nAligned Motifs (a1...an):")
    for i, motif in enumerate(final_motifs):
        if target_motif:
            matches = sum(1 for a, b in zip(motif, target_motif) if a == b)
            match_percentage = (matches / l_mer_length) * 100
            print(f"S{i+1} (Pos {best_positions[i]:>2}): {motif}  -> {match_percentage:.0f}% match to {target_motif}")
        else:
            print(f"S{i+1} (Pos {best_positions[i]:>2}): {motif}")

Gibbs Sampler - Assignment Build
Do you want to type your own sequences manually? If not the code will use a test sequence.. (y/n): y

Enter your sequences one by one [Use only A, T, G, C].
Type 'DONE' when you are finished.
Enter Sequence: AGAGATCTCAGA
Enter Sequence: ACACTAGCTACGT
Enter Sequence: AGCTAGTTAA
Enter Sequence: AATGATGCAC
Enter Sequence: DONE

Enter the target motif if you know it (or hit ENTER to skip): ATATAGCA

Initiating Gibbs Sampler for 8-mers...
STEP-BY-STEP BREAKDOWN (First Iteration)
Goal: Count residue occurrences for each position for all unchosen sequences.
• AGAGATCTCAGA  <-- EXCLUDED (Target for sampling)
• ACACT[AGCTACGT]
• [AGCTAGTT]AA
• [AATGATGC]AC

Raw Count Matrix (Matches lecture slides before pseudocounts):
| Nt | bg0 |  1  |  2  |  3  |  4  |  5  |  6  |  7  |  8  |
------------------------------------------------------------
|  A |  5  |  3  |  1  |  0  |  0  |  3  |  0  |  0  |  0  |
|  C |  3  |  0  |  0  |  2  |  0  |  0  |  1  |  0  |  1  |
|  

### References & Acknowledgements

* **Primary Algorithm Source:** Rouchka, E. C. (2008). *A Brief Overview of Gibbs Sampling*. [ResearchGate Link](https://www.researchgate.net/publication/2336786_A_Brief_Overview_of_Gibbs_Sampling)
* **Biological Context & Framework:** Oregon State University. *Applied Bioinformatics - Chapter 2: Sequence Motifs*. [OER Link](https://open.oregonstate.education/appliedbioinformatics/chapter/chapter-2-sequence-motifs/)
* **Youtube:** [Youtube Link](https://youtu.be/KqquwB_kHvU)

### Algorithm Considerations & Drawbacks
Based on the lecture notes and the actual way this code runs, there are some tradeoffs we have to accept:
* **The Good:** The core algorithm runs in O(N) linear run-time. Because of the stochastic roulette-wheel sampling, it actively avoids the local optimum problem. We can also view suboptimal results if we want to trace the iterations.
* **The Bad (Algorithm Limits):** It requires relatively large sets of sequences to work properly. Even with our probabilistic sampling, it is not guaranteed to reach an optimal value. Also, we must specify an estimate for the motif width upfront. If the actual biological motif is 12 bases long and I tell the code to look for an $l$-mer of 6, it completely misses the big picture.
* **The Drawbacks of My Specific Code:** Because I wrote this using native Python dictionaries and nested loops to make the step-by-step math printable for the manual trace, it's pretty unoptimized for speed. If we threw a massive real-world genomic dataset at this, it would lag terribly compared to a vectorized NumPy/C++ version. Also, to fight the fact that Gibbs isn't actually guaranteed to reach the optimal value, I brute-forced it by adding a massive `num_restarts` outer loop, which multiplies the run-time heavily just to secure a safe grade.

### Deep Learning Crossovers (Why the code is structured this way)
While this is a bioinformatics algorithm, I actually ported over several concepts directly from deep learning to optimize how the Gibbs Sampler converges:

1. **Early Stopping / Patience (`patience_counter`):** Instead of looping blindly for $N$ iterations, I implemented an early stopping callback. If the model's convergence score ($L$) fails to improve after 50 continuous epochs (iterations), training halts. This prevents overfitting to a local motif and saves compute time.
2. **Laplace Smoothing (Pseudocounts):** I added `+1.0` to the background and profile matrices. In NLP/Deep Learning embeddings, an unseen token will crash the model with a zero-probability multiplication. Slapping a pseudocount here ensures our mathematical chain doesn't break if a sequence is missing a specific nucleotide.
3. **Log-Likelihood (`math.log2`):** Multiplying raw probabilities over and over causes severe floating-point underflow (eventually just rounding to 0.0 in Python). Using Log-Odds allows us to *add* scores instead, identical to how neural networks use Cross-Entropy or LogSoftmax loss functions for stability.
4. **Softmax-style Sampling (Roulette Wheel):** Gibbs sampling gets trapped easily if you just use `argmax` (greedy decoding) to pick the best l-mer. I normalized the weights and sampled probabilistically—essentially the same logic as using a temperature parameter in Softmax to allow the model to "explore" alternative alignments.
5. **Random Weight Initialization (`num_restarts`):** A bad random start means getting stuck in a local minimum. Just like re-initializing weights in Stochastic Gradient Descent, running multiple random restarts guarantees the algorithm eventually finds the global optimal motif.

### Integration with Previous Bioinformatics Coursework
To tie this into the past assignments, the core of this algorithm builds directly on foundational bioinformatics concepts taught in class so far:
1. **Position Weight Matrices (PWMs / PSSMs):** The `create_profile` function is essentially just building a PWM. We tally the raw frequencies of the motif, apply pseudocounts, and normalize them into probabilities to score unseen sequences, exactly like we did in the basic motif evaluation labs.
2. **Gapless Local Multiple Sequence Alignment (MSA):** Unlike exact dynamic programming methods (like Needleman-Wunsch or Smith-Waterman) used for pairwise alignment, this Gibbs sampler performs a heuristic, stochastic *local* alignment across multiple sequences simultaneously, strictly without gaps.
3. **Zero-Order Background Model:** The `get_background` function calculates a global probability for A, C, G, and T. This assumes each nucleotide's appearance is independent of its neighbors, serving as our zero-order Markov null model to test the motif's statistical significance against.



> **Note:** AI (Google Gemini) was used as a coding assistant to translate the course's mathematical formulas and pseudocode into Python. All core algorithm logic is derived strictly from the referenced course materials.